In [ ]:
# !pip install transformers

In [ ]:
# !pip install transformers[torch]

In [ ]:
# %pip install -U jupyter ipywidgets tqdm

In [7]:
import pandas as pd
import re
import torch
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [8]:
train_data = pd.read_csv("X:\\VScode\\Artificial_intelligence_n_Machine_learning\\Deep_Learning\\text_summarizer\\CSV\\samsum-train.csv")
validation_data = pd.read_csv("X:\\VScode\\Artificial_intelligence_n_Machine_learning\\Deep_Learning\\text_summarizer\\CSV\\samsum-validation.csv")

In [9]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [10]:
print(f"Shape: {train_data.shape}")
train_data.info()
validation_data.info()

Shape: (14732, 3)
<class 'pandas.DataFrame'>
RangeIndex: 14732 entries, 0 to 14731
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        14732 non-null  str  
 1   dialogue  14731 non-null  str  
 2   summary   14732 non-null  str  
dtypes: str(3)
memory usage: 9.2 MB
<class 'pandas.DataFrame'>
RangeIndex: 818 entries, 0 to 817
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        818 non-null    str  
 1   dialogue  818 non-null    str  
 2   summary   818 non-null    str  
dtypes: str(3)
memory usage: 514.0 KB


In [11]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [12]:
# Random Sampling

train_data = train_data.sample(n=14500, random_state=42).reset_index(drop=True)
validation_data = validation_data.sample(n=800, random_state=42).reset_index(drop=True)

In [13]:
# Data Pre-Processing

In [14]:
def clean_data(text):
    if pd.isna(text):
        return ""
    text = re.sub(r"\r\n"," ",text) # lines
    text = re.sub(r"\s+"," ",text) # space
    text = re.sub(r"<.*?>"," ",text) # tags
    text = text.strip().lower()
    return text

In [15]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

validation_data["dialogue"] = validation_data["dialogue"].apply(clean_data)
validation_data["summary"] = validation_data["summary"].apply(clean_data)

In [16]:
# Token

In [17]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")


In [18]:
def tokenize(data):
    inputs = tokenizer(
        data["dialogue"],
        padding="max_length",
        max_length=512,
        truncation=True
    )

    targets = tokenizer(
        data["summary"],
        padding="max_length",
        max_length=180,
        truncation=True
    )

    inputs["labels"] = targets["input_ids"]
    return inputs

In [19]:
train_data_set = train_data.apply(tokenize, axis=1).tolist()
val_data_set = validation_data.apply(tokenize, axis=1).tolist()


In [20]:
print(f"Length of token for traning dataset: {len(train_data_set[0]['input_ids'])}")
print(f"Length of token for validation dataset: {len(val_data_set[0]['labels'])}")

Length of token for traning dataset: 512
Length of token for validation dataset: 180


In [21]:
# Model

In [22]:
T5model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [23]:
# Fine tuning
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device: {device}")
T5model.to(device)

Device: cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [24]:
Training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs = 6,
    weight_decay = 0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=500
)

In [25]:
trainer = Trainer(
    model = T5model,
    args=Training_args,
    train_dataset=train_data_set,
    eval_dataset=val_data_set
) 

In [26]:
# train model

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.328338,0.284120
2,0.305223,0.273960
3,0.290598,0.271677
4,0.290100,0.268288
5,0.283669,0.267629
6,0.278570,0.267358


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10878, training_loss=0.44595589754357456, metrics={'train_runtime': 2863.8275, 'train_samples_per_second': 30.379, 'train_steps_per_second': 3.798, 'total_flos': 1.1774736728064e+16, 'train_loss': 0.44595589754357456, 'epoch': 6.0})

In [27]:
# save the trained model

In [28]:
T5model.save_pretrained('./Saved_Model')
tokenizer.save_pretrained('./Saved_Model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./Saved_Model\\tokenizer_config.json', './Saved_Model\\tokenizer.json')

In [29]:
model = T5ForConditionalGeneration.from_pretrained('./Saved_Model')
tokenizer = T5Tokenizer.from_pretrained('./Saved_Model')
model.to(device)


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [30]:
# test model

In [31]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean input
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        return_tensors='pt'
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}
    target = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 128,
        num_beams = 4,
        early_stopping = True     
    )
    summary = tokenizer.decode(target[0], skip_special_tokens=True)
    return summary


In [32]:
test_diagogue = '''
Sam: "Are you using that new AI data assistant for your analytics dashboard?"

Taylor: "Yeah, I hooked it up to our SQL database. It writes complex aggregation queries just from plain English prompts."

Sam: "Does it handle messy data well, or do you still have to clean everything beforehand?"

Taylor: "It surprisingly catches missing values and outliers, though I still double-check the preprocessing steps."

Sam: "That sounds like a massive time-saver for exploratory data analysis."

Taylor: "Definitely. It cuts down hours of manual coding into a couple of minutes."
'''

summary = summarize_dialogue(test_diagogue)

In [33]:
print(f"Summary:\n{summary}")

Summary:
taylor hooked the new ai data assistant to his sql database. it writes complex aggregation queries from plain english prompts.
